In [ ]:
import numpy as np
import pandas as pd

import os
import math
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.neighbors import NearestNeighbors
from itertools import product

cityTable = pd.read_csv('city_attributes.csv')
temperatureDF = pd.read_csv('temperature.csv', index_col=0)

temperatureDF.index = pd.to_datetime(temperatureDF.index)

In [ ]:
temperatureDF

In [ ]:
def takensEmbedding(data, delay, dimension):
    if delay*dimension > len(data):
        raise ValueError("Delay times dimension exceed length of data!")
    embeddedData = np.array([data[0:len(data)-delay*dimension]])
    for i in range(1, dimension):
        embeddedData = np.append(embeddedData, 
                                [data[i*delay:len(data)-delay*(dimension-i)]], 
                                 axis=0)
      
        # if delay*dimension < len(data) then extra data points are removed.
    return embeddedData

In [ ]:
t = pd.date_range(pd.to_datetime('22/6/2015', dayfirst=True),
                  pd.to_datetime('31/8/2015', dayfirst=True), 
                  freq = 'h')

weatherDataMontreal = temperatureDF.loc[t, 'Montreal']
originalSignal = weatherDataMontreal;

# removing monthly and yearly dynamics by applying rolling mean over one day and plot the signal 
# (low pass filter)

windowSize = 24
lowPassFilteredSignal = weatherDataMontreal.rolling(windowSize, center=True).mean()
weatherDataMontreal = weatherDataMontreal - lowPassFilteredSignal
weatherDataMontreal = weatherDataMontreal.dropna()

# 2D embedding
embeddedWeather = takensEmbedding(weatherDataMontreal, 5, 2)

fig, ax = plt.subplots(nrows=3, ncols=1, figsize=(15,14))
ax[0].plot(weatherDataMontreal)
ax[1].plot(embeddedWeather[0,:],embeddedWeather[1,:])
ax[2].axis('off')

# 3 D embedding
embeddedWeather3 = takensEmbedding(weatherDataMontreal, 6,3);

#plot the 3D embedding
ax = fig.add_subplot(3, 1, 3, projection='3d')
ax.plot(embeddedWeather3[0,:],embeddedWeather3[1,:],embeddedWeather3[2,:]);

In [ ]:
print(embeddedWeather3)

## Finding optimal Delay value.

### Mutual Information 


The interval between $[x_{min},x_{max}]$ is divided into a large number of bins. Denote by $P_{k}$ the probability of an element of the time-series to be in the $k^{th}$ bin and by $P_{h,k}(\tau)$ the probability that $x_{i}$ is the $h^{th}$ bin while $x_{i+\tau}$ is in the $k^{th}$ bin.


$$
I(\tau) = - \sum_{h=1}^{nBins} \sum_{k=1}^{nBins} P_{h,k}(\tau) \log \frac{P_{h,k}(\tau)}{P_h P_k}.
$$

The first minimum of $I(\tau)$ as a function of $\tau$ gives the optimal delay, since there we get the largest information by adding x

In [ ]:

# def calculate_mutual_information(data, delay, nBins):
#     xmin = data.min()
#     xmax = data.max()

#     input = data[delay:len(data)]
#     delayed_input = data[0:len(data)-delay]

#     xbounds = np.linspace(xmin, xmax, nBins+1)

#     pk = {key+1 : 0 for key in range(len(xbounds))}
#     ph = {key+1 : 0 for key in range(len(xbounds))}
#     phk = {key : 0 for key in list(product(range(len(xbounds)), range(len(xbounds))))}
#     N = len(input)

#     for x,xt in  zip(input, delayed_input):
#         for k in range(1, len(xbounds)):
#             if xbounds[k-1] <= x < xbounds[k]:
#                 pk[k] += 1/N
#                 for h in range(1, len(xbounds)):          # h inside k: correct for phk
#                     if xbounds[h-1] <= xt < xbounds[h]:
#                         phk[(h, k)] += 1/N
                
#         for h in range(1, len(xbounds)):
#                 if xbounds[h-1] <= xt < xbounds[h]:
#                     ph[h] += 1/N


#     I = -1*sum(phk[key] * np.log(phk[key] / (ph[key[0]] * pk[key[1]]))
#         for key in list(product(range(len(xbounds)), range(len(xbounds))))
#         if phk[key] > 0)

#     return I

In [ ]:
def calculate_mutual_information(data, delay, nBins):
    
    N = len(data)-delay
    x = data[delay:]
    x_t = data[:N]

    # print("x", x)
    # print("x_t", x_t)

    edges = np.linspace(data.min(),data.max(), nBins+1)
    # print('edges', edges)

    k_idx = np.clip(np.digitize(x, edges), 1, nBins)
    h_idx = np.clip(np.digitize(x_t,edges), 1, nBins)

    pk  = np.zeros(nBins)
    ph  = np.zeros(nBins)
    phk = np.zeros((nBins, nBins))

    for i in range(N):
        pk[k_idx[i]-1] += 1/N # bins start from idx 1 to 10. And python array idx 0 to 9.
        ph[h_idx[i]-1] += 1/N
        phk[h_idx[i]-1, k_idx[i]-1] += 1/N

    I = sum(phk[h,k] * np.log(phk[h,k]/ (ph[h]*pk[k])) for h in range(nBins) for k in range(nBins) if phk[h,k] > 0)  


    return I





In [ ]:
# def calculate_mutual_information(data, delay, nBins):
#     # Split the time series based on delay
#     x = data[delay:]
#     x_t = data[:-delay] # Cleaner syntax for len(data) - delay
    
#     # 1. Compute 2D and 1D histograms efficiently
#     # np.histogram2d automatically handles bin edge limits safely
#     phk, x_edges, y_edges = np.histogram2d(x_t, x, bins=nBins)
    
#     # 2. Convert counts into probabilities
#     N = len(x)
#     phk /= N
#     ph = np.sum(phk, axis=1) # Marginal probability of x_t
#   i  pk = np.sum(phk, axis=0) # Marginal probability of x
    
#     # 3. Create a mask to avoid log(0) errors
#     nonzero = phk > 0
    
#     # 4. Compute Mutual Information (Notice: No negative sign)
#     # Use np.outer to multiply marginal probabilities across the grid
#     joint_prob = phk[nonzero]
#     marginal_product = np.outer(ph, pk)[nonzero]
    
#     I = np.sum(joint_prob * np.log(joint_prob / marginal_product))
    
#     return I

In [ ]:
# def calculate_mutual_information(data, delay, nBins):
#     "This function calculates the mutual information given the delay"
#     I = 0;
#     xmax = max(data);
#     xmin = min(data);
#     delayData = data[delay:len(data)];
#     shortData = data[0:len(data)-delay];
#     sizeBin = abs(xmax - xmin) / nBins;
#     #the use of dictionaries makes the process a bit faster
#     probInBin = {};
#     conditionBin = {};
#     conditionDelayBin = {};
#     for h in range(0,nBins):
#         if h not in probInBin:
#             conditionBin.update({h : (shortData >= (xmin + h*sizeBin)) & (shortData < (xmin + (h+1)*sizeBin))})
#             probInBin.update({h : len(shortData[conditionBin[h]]) / len(shortData)});
#         for k in range(0,nBins):
#             if k not in probInBin:
#                 conditionBin.update({k : (shortData >= (xmin + k*sizeBin)) & (shortData < (xmin + (k+1)*sizeBin))});
#                 probInBin.update({k : len(shortData[conditionBin[k]]) / len(shortData)});
#             if k not in conditionDelayBin:
#                 conditionDelayBin.update({k : (delayData >= (xmin + k*sizeBin)) & (delayData < (xmin + (k+1)*sizeBin))});
#             Phk = len(shortData[conditionBin[h] & conditionDelayBin[k]]) / len(shortData);
#             if Phk != 0 and probInBin[h] != 0 and probInBin[k] != 0:
#                 I -= Phk * math.log( Phk / (probInBin[h] * probInBin[k]));
#     return I;

In [ ]:
datDelayInformation = []
for i in range(1,21):
    datDelayInformation = np.append(datDelayInformation,[calculate_mutual_information(weatherDataMontreal,i,16)])
print(datDelayInformation)


In [ ]:
import matplotlib.pyplot as plt

plt.plot(range(1,21),datDelayInformation);
plt.xlabel('delay');
plt.ylabel('mutual information');

## Finding Optimal Dimension

### False Nearest Neighbors

- Used to determine the correct embedding dimension.
- Points which are closer in one embedding dimension should be close in next embedding dimension.


- If the embedding dimension is small, two points may appear overalapping or close to each other.
- To ensure that two points are indeed nearest neighbors, we would perform False Nearest Neighbors test. 
- The algorithm looks at two nearest neighbors in $d^{th}$ dimension and project it to $d+1^{th}$ dimension. 
- If the distance between two these data points drastically increases in higher dimension, the original neighbor was false.
- The ideal embedding dimension is reached when the percentage of False Nearest Neighbors drops to zero.


- In time-delay embedding, a vector in $d$ dimension is represented as: $y_{i} = [x(i),x(i+\tau),x(i+2\tau)\cdots, x(i+(d-1)\tau)]$.
- A vector in $d+1^{th}$ dimension is given by,  $y_{i} = [x(i),x(i+\tau),x(i+2\tau)\cdots, x(i+(d-1)\tau), x(i+d\tau)]$

$$

\frac{|x(i+d\tau)-x(n(i)+d\tau)|}{R_{d}(i)} > R_{tol}

$$

where $R_{d}(i) = \|y_{i}(d) - y_{n(i)}(d)\|$. The threshold value $R_{tol}$ is set between $10$ and $15$.

In [ ]:
embeddedData = takensEmbedding(data = range(10), delay=2, dimension=3)
print(embeddedData)

In [ ]:
# Check only the nearest neighbor
nbrs = NearestNeighbors(n_neighbors=2, algorithm='auto').fit(embeddedData.transpose())

# calculate the distances
distances, indices = nbrs.kneighbors(embeddedData.transpose())
print("distances\n", distances)
print("idx\n", indices)

- **distances** - A matrix showing the mathematical distances from each data point to its nearest neighbors.
- indice --> `[0,1]` implies Point 0's closest neighbor is Point 1.
         --> `[1,0]` implies Point 1's closest neighbor is Point 0   


Because you set `n_neighbors=2`, both outputs are arrays with a shape of `(number of data points, 2)`

In [ ]:
epsilon = np.std(distances.flatten())
print(epsilon)

In [ ]:
nFalseNN = 0
data = range(10)
delay =2 
dimension=3

for i in range(0, len(data)-delay*(dimension+1)):
    r = abs(data[i+dimension*delay] - data[indices[i,1]+dimension*delay]/ distances[i,1])
    print(r)
    if r > 10:
        nFalseNN += 1;

nFalseNN 

`nFalseNN = 0` implies that there are no False Neighbors.

In [ ]:
def false_nearest_neighours(data,delay,embeddingDimension):

    embeddedData = takensEmbedding(data,delay,embeddingDimension)
    nbrs = NearestNeighbors(n_neighbors=2, algorithm='auto').fit(embeddedData.transpose())
    distances, indices = nbrs.kneighbors(embeddedData.transpose())
    epsilon = np.std(distances.flatten())
    nFalseNN = 0
    
    for i in range(0, len(data)-delay*(embeddingDimension+1)):
        if (0 < distances[i,1] < epsilon) and ( (abs(data[i+embeddingDimension*delay] - data[indices[i,1]+embeddingDimension*delay]) / distances[i,1]) > 10):
            nFalseNN += 1;
    
    return nFalseNN



In [ ]:
nFNN = []
for i in range(1,7):
    nFNN.append(false_nearest_neighours(weatherDataMontreal,1,i) / len(weatherDataMontreal))
plt.plot(range(1,7),nFNN);
plt.xlabel('embedding dimension');
plt.ylabel('Fraction of fNN');

## Multivariate time-delay embedding

In [ ]:
data = np.random.random(size=(4,3))
data

In [ ]:
data_t = data.transpose()

data_t

In [ ]:
ldim, num_features =  data.shape
ldim, num_features

In [ ]:
delay = 1
new_ldim = 4

In [ ]:
data = np.random.random(size=(4,3))
data_t = data.transpose()

ldim, num_features =  data.shape

delay = 1
new_ldim = 4


temp = []
for j in range(new_ldim):
    row = np.concatenate([data_t[i][j:delay+j+1]  for i in range(num_features)])
    if len(row) == (delay+1)*num_features:
        temp.append(row)
temp

In [ ]:
data = np.random.random(size=(4,3))
data

In [ ]:

# data_t = data.transpose()

ldim, num_features =  data.shape

delay = 2
dimension = 2


embeddedData = []
for i in range(ldim):
    row = np.concatenate(data[i*delay:i+dimension*delay])
    if len(row) == num_features*dimension:
        embeddedData.append(row)
    
np.array(embeddedData)


# embeddedData = np.arra

# for i in range(ldim):
#      print(data[i:i+dimension:delay])

# np.hstack([np.concatenate([data[i:i+dimension:delay] for i in range(ldim)])])



# temp = []
# # for j in range:
# #     # row = np.concatenate([data_t[i][j:delay+j+1]  for i in range(num_features)])
# #     row = np.concatenate(data[])
# #     if len(row) == (delay+1)*num_features:
# #         temp.append(row)


# embeddedData

In [ ]:
data = np.random.random(size=(1,3))
data

In [ ]:
import numpy as np

ldim, num_features = data.shape
delay = 1
dimension = 2

# Calculate the exact number of valid rows before out-of-bounds truncation
max_i = ldim - (dimension - 1) * delay

embeddedData = np.hstack([data[j * delay : max_i + j * delay] for j in range(dimension)])
embeddedData

In [ ]:

delay = 1
dimension = 2
max_i = ldim - (dimension - 1) * delay
for j in range(dimension):
    print(j)
    print(j*delay)
    print(max_i + j * delay)
    print(data[j * delay : max_i + j * delay])

In [ ]:
# print()
# new_data = np.concatenate(
#     [new_data, data[i*delay*new_ldim : (i+1)*delay*new_ldim]],
#     axis=1
# )

# new_data = np.append(new_data, [temp])

# new_data = np.

# np.concatenate(data[i*delay*dimension : (i+1)*delay*dimension])
# temp = np.concatenate(data[i*delay*(new_ldim):(i+1)*delay*(new_ldim)])
# print(temp)

In [ ]:
new_data = []
for i in range(new_ldim):
    new_data.append(np.concatenate(data[i*delay*(new_ldim):(i+1)*delay*(new_ldim)]))


embeddedData = np.array(new_data)
embeddedData

In [ ]:
new_data

In [ ]:
dimension = 3

embeddedData = data[0 : new_ldim - delay*dimension]        # shape (N-d*τ, 3)
for i in range(1, dimension):
    embeddedData = np.concatenate(
        [embeddedData, data[i*delay : new_ldim - delay*(dimension-i)]], 
        axis=1   # stack components, not time steps
    )
embeddedData

In [ ]:
dimension = 2

N, num_features = data.shape
n_points = N - delay * dimension  # number of embedded vectors

embeddedData = np.hstack([
    data[i*delay : i*delay + n_points]   # shape (n_points, num_features)
    for i in range(dimension)
])

embeddedData

In [ ]:
n_points

In [ ]:
def delay_embedding(data, d, tau):
    N = len(data)
    indices = np.arange(d) * tau + np.arange(N - (d - 1) * tau)[:, None]
    print(indices)
    embedded_data = data[indices]
    return embedded_data

In [ ]:
arr = np.array([1,2,3,4])
emb = delay_embedding(arr,2,1)
emb


In [ ]:
test = np.array([[1,2,3],[2,4,4]])
N = test.shape[0]

In [ ]:
N

In [ ]:
def takensEmbeddingMultivariate(data, dimension, delay):

    N = data.shape[0]

    max_rows = N - (dimension - 1) * delay

    embeddedData = np.hstack([data[j * delay : max_rows + j * delay] for j in range(dimension)])

    return embeddedData


In [ ]:
data = np.random.random(size=(4,3))
takensEmbeddingMultivariate(data, dimension=2, delay=1)

In [ ]:
    indices = np.arange(dimension) * delay + np.arange(N - (dimension - 1) * delay)[:, None]

In [ ]:
def delay_embedding(data, d, tau):
    N = len(data)
    indices = np.arange(d) * tau + np.arange(N - (d - 1) * tau)[:, None]
    print(indices)
    embedded_data = data[indices]
    return embedded_data

In [ ]:
delay_embedding(data,2, 1)

In [ ]:
data = np.arange(0,15,1).reshape(5,3)
data

In [ ]:
takensEmbedding(data,dimension=2, delay=2)